In [1]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

train_df = pd.read_csv("preprocessed_train_final.csv")


c:\Users\ilker\anaconda3\envs\torchgpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_df = train_df.groupby('category_fixed', ).sample(n=300, replace=True).reset_index(drop=True)

In [3]:
import numpy as np
import pandas as pd
import re
from tqdm.auto import tqdm

TEXT_COL = "text_for_model" 

dfB = train_df[["product_id", TEXT_COL]].copy()
dfB[TEXT_COL] = dfB[TEXT_COL].fillna("").astype(str)

In [4]:
import torch
from sentence_transformers import SentenceTransformer

MODEL_NAME = "Trendyol/TY-ecomm-embed-multilingual-base-v1.2.0"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(MODEL_NAME, device=device, trust_remote_code=True)
dim = model.get_sentence_embedding_dimension()

print("Device:", device)
print("Embedding dim:", dim)


Device: cuda
Embedding dim: 768


In [5]:
import os

EMB_PATH = "embeddings_float16.memmap"
N = len(dfB)

# float16 saves disk & speeds IO; OK for clustering
emb = np.memmap(EMB_PATH, dtype="float16", mode="w+", shape=(N, dim))

BATCH = 4096 if device == "cuda" else 1024

# E5 prefers a prefix; use "passage: " for texts
texts = ("passage: " + dfB[TEXT_COL]).tolist()

for start in tqdm(range(0, N, BATCH), desc="Embedding"):
    end = min(start + BATCH, N)
    batch_text = texts[start:end]

    # normalize_embeddings=True gives unit vectors (good for cosine/IP clustering)
    vec = model.encode(
        batch_text,
        batch_size=min(256, BATCH),
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype(np.float16)

    emb[start:end] = vec

emb.flush()
print("Saved memmap:", EMB_PATH, "shape:", emb.shape)


Embedding: 100%|██████████| 84/84 [09:48<00:00,  7.00s/it]


Saved memmap: embeddings_float16.memmap shape: (342000, 768)


In [6]:
import faiss
import numpy as np

K = 5
SEED = 42

emb = np.memmap(EMB_PATH, dtype="float16", mode="r", shape=(N, dim))

SAMPLE_N = min(500_000, N)
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(N, size=SAMPLE_N, replace=False)

X = emb[sample_idx].astype(np.float32)  # faiss wants float32
print("KMeans train sample:", X.shape)

# Faiss KMeans (uses inner product if vectors are normalized -> cosine-like)
kmeans = faiss.Kmeans(
    d=dim,
    k=K,
    niter=20,
    nredo=2,
    verbose=True,
    seed=SEED
)
kmeans.train(X)

centroids = kmeans.centroids  # float32, shape (K, dim)
print("Centroids:", centroids.shape)


KMeans train sample: (342000, 768)
Centroids: (5, 768)


In [7]:
# Build index for nearest-centroid search
index = faiss.IndexFlatIP(dim)  # dot product; with normalized vectors -> cosine similarity
index.add(centroids.astype(np.float32))

cluster_id = np.empty(N, dtype=np.int16)

ASSIGN_BATCH = 200_000  # tune to your RAM
for start in tqdm(range(0, N, ASSIGN_BATCH), desc="Assign clusters"):
    end = min(start + ASSIGN_BATCH, N)
    Xb = emb[start:end].astype(np.float32)
    sims, ids = index.search(Xb, 1)
    cluster_id[start:end] = ids[:, 0].astype(np.int16)

dfB["cluster_id"] = cluster_id
dfB["cluster_id"].value_counts().sort_index()


Assign clusters: 100%|██████████| 2/2 [00:00<00:00,  2.20it/s]


cluster_id
0    80448
1    64261
2    81158
3    49508
4    66625
Name: count, dtype: int64

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

# sample per cluster for keyword extraction
PER_CLUSTER_SAMPLE = 100_000
rng = np.random.default_rng(SEED)

sample_rows = []
for cid in range(K):
    idx = np.where(dfB["cluster_id"].values == cid)[0]
    take = min(PER_CLUSTER_SAMPLE, len(idx))
    pick = rng.choice(idx, size=take, replace=False)
    sample_rows.append(dfB.iloc[pick])

kw_df = pd.concat(sample_rows, ignore_index=True)

# domain stopwords (add freely)
unit_stop = {
    "cm","mm","ml","l","lt","gr","g","kg","mg","adet","pcs","pc",
    "x","li","lü","lu","lı","set","paket","kutu","boy","ebat"
}

# optional: Turkish stopwords (small set; expand if you want)
tr_stop = {
    "ve","ile","icin","için","da","de","bir","bu","şu","o","çok",
    "en","mi","mı","mu","mü","ya","yada","veya","gibi","olarak"
}

COLORS = {
    "siyah","beyaz","mavi","kirmizi","yesil","gri","mor","pembe","turuncu","kahverengi","lacivert","bej","bordo",
    # ink/toner colors
    "sari","cyan","magenta","macenta","mavi","black","yellow"
}

MARKETING = {
    "orijinal","orjinal","muadil","premium","luxury","golden","series","sert","seffaf","desenli",
    "kampanya","indirim","firsat","ucretsiz","kargo","bedava","yeni","sezon","kaliteli"
}

GENERIC = {
    "icin","for","ve","ile","uyumlu","set","paket","takim","adet","parca","x","x2","x3","x4","x5"
}


stopwords = unit_stop | tr_stop | COLORS | MARKETING | GENERIC

# TF-IDF on sampled texts
# word-level + Turkish chars already in canon_text
tfidf = TfidfVectorizer(
    min_df=5,
    max_df=0.6,
    ngram_range=(1,2),
    max_features=200_000,
    stop_words=list(stopwords),
    token_pattern=r"(?u)\b[^\W\d_][^\W_]+\b"
)
X_tfidf = tfidf.fit_transform(kw_df[TEXT_COL])
vocab = np.array(tfidf.get_feature_names_out())

# compute top terms per cluster
cluster_titles = {}
cluster_keywords = {}

for cid in range(K):
    m = (kw_df["cluster_id"].values == cid)
    # mean tfidf per term for this cluster
    mean_vec = X_tfidf[m].mean(axis=0).A1
    top_idx = np.argsort(-mean_vec)[:20]
    top_terms = vocab[top_idx].tolist()
    cluster_keywords[cid] = top_terms

    # simple title heuristic: first 3–5 terms joined
    # (you can later rename manually to business department names)
    cluster_titles[cid] = " / ".join(top_terms[:10])

cluster_titles


{0: 'makinesi / erkek / led / oto / pro / usb / oyuncu / seti / ekran / model',
 1: 'takımı / ahşap / kişilik / seti / bebek / çocuk / dekoratif / örtüsü / yatak / odası',
 2: 'seti / kalem / saç / bakım / boya / fırçası / kremi / makyaj / temizleme / kalemi',
 3: 'kedi / organik / çay / doğal / bebek / poşet / yağı / yeşil / köpek / yapımı',
 4: 'kadın / erkek / beden / büyük beden / büyük / takım / detaylı / tesettür / taşlı / yaka'}

In [9]:
def show_cluster(cid, n_examples=12):
    print(f"\n=== Cluster {cid} ===")
    print("Size:", int((dfB["cluster_id"] == cid).sum()))
    print("Keywords:", ", ".join(cluster_keywords[cid][:15]))
    ex = dfB[dfB["cluster_id"] == cid].sample(n=min(n_examples, (dfB["cluster_id"] == cid).sum()), random_state=SEED)
    display(ex[["product_id", TEXT_COL]].head(n_examples))

for cid in range(K):
    show_cluster(cid, n_examples=10)



=== Cluster 0 ===
Size: 80448
Keywords: makinesi, erkek, led, oto, pro, usb, oyuncu, seti, ekran, model, bluetooth, araç, kırmızı, metal, deri


,product_id,text_for_model
190345,198816939,çarşambaca kovir misin agam büyük boy mouse pad
170392,138757266,te 70atc avr 14 5j kırıcı delici 10kg
79489,65990305,bdlp 049 swing 3 kapılı 2 çekmeceli dolap beyaz
19101,34006916,yankı ördek düdüğü
159938,78483500,galaxy a40 oyuncu kulaklığı 3 5 mm girişli
5718,35448620,huawei watch gt active rainbow sport band kord...
13935,100373082,audi a3 coupe yan marşpiyel sağ sol 2006 2012 ...
208806,82035304,krank kasnagı vw polo golf bora 1 9sdı 1 9tdı ...
340921,169464820,atlantis şifonyer ve aynası
103461,7169691,canon eos 7d için phottix battery grip bg e7



=== Cluster 1 ===
Size: 64261
Keywords: takımı, ahşap, kişilik, seti, bebek, çocuk, dekoratif, örtüsü, yatak, odası, çift, metal, mutfak, masa, tek


,product_id,text_for_model
46150,40969023,alya berjer
221415,214672091,3 katlı patates soğanlık patates soğan sepeti ...
210086,45703167,alex sek sek halısı kızlar için
25643,119162586,doğal ahşap katlanabilir 120 x 70 cm bahçe bal...
334954,167845001,tuz karabiber değirmeni 2 li set ceviz 14 cm
76119,125825880,nazar boncuklu araba süsü dikiz aynası araba s...
258039,143712554,dede spiderman büyük kova set
19333,40455497,kontes 8 li eskitme avize 8xe14
220905,171327210,istanbul fil serisi 6 lı pasta tabağı 15cm
93869,95208120,royal shop banyo yaptırılan mavi banyo zamanı ...



=== Cluster 2 ===
Size: 81158
Keywords: seti, kalem, saç, bakım, boya, fırçası, kremi, makyaj, temizleme, kalemi, yüz, no, sprey, renk, boyası


,product_id,text_for_model
339202,164179591,thıckenıng shampoo 250ml
314272,3270304,güneş koruyucu yüz kremi mineral sunscreen spf...
119190,130665876,melisa yağı işıklı su hazneli hava nemlendiric...
320722,73936176,bej klor bazlı çamaşır suyu ağartıcı 20 kg
294880,47147626,slider edge xb tükenmez kalem kırmızı 5 li
340293,110360011,marvel şekerli kağıda baskı
69571,135815488,stop vx yüz yenileme yeniden şekillendirme gen...
22289,99554777,sir ağda bandı 24 lü klasik x5 adet
96005,53585058,suya dayanıklı siyah waterproof dipliner 5 ml ...
306063,90801761,süper japon yapıştırıcı 1 adet 20 gr



=== Cluster 3 ===
Size: 49508
Keywords: kedi, organik, çay, doğal, bebek, poşet, yağı, yeşil, köpek, yapımı, karışık, aromalı, ev yapımı, maması, ev


,product_id,text_for_model
302626,160908141,kalsiyum magnezyum vitamin d3 çinko içeren gıd...
228336,89795746,propolis sprey 2 li 50 ml
340266,74014497,bakır işlemeli şekerlik bakır lokumluk bakır ç...
95311,92836321,250 cc sızdırmaz gıda kabı pet 1000 adet
308151,154657179,kitten milk powder yavru kediler için süt tozu...
124518,43901058,kudüs hurması jumbo 1000 gram
139203,53767263,100 doğal yeşil bıttım sabunu 575 gr
39932,52321963,lavanta yağlı masaj ve temizlik havlusu 12x40 ...
168685,148326419,dr clauder köpek ödül country dental balıklı 1...
178283,32586972,mama mia mama sandalyesi



=== Cluster 4 ===
Size: 66625
Keywords: kadın, erkek, beden, büyük beden, büyük, takım, detaylı, tesettür, taşlı, yaka, deri, çantası, eşofman, gümüş, uzun


,product_id,text_for_model
91973,45021320,kadın ten sütyen 9500
56124,89154092,kadın ekru nakışlı dantelli ekru bornoz takımı...
29652,194937468,super pug baskılı cerrahi doktor ve hemşire bo...
100020,126423438,kadın ağı açık g strinğ
50292,34233859,kadın turuncu bikini altı 9yak88784bm
63044,175352522,büyük beden kapşonlu somon hırka
64409,37851000,büyük beden beli lastikli cepli bol paça kadın...
69250,2746973,kadın gold choo choker
106581,138665614,ip askılı gecelik 3219 leopar
227222,112060214,kırmızı bandanası ile hayal eden kadın desen b...


In [10]:
import numpy as np
import pandas as pd

# dfB should exist from Part B and include: product_id, TEXT_COL, cluster_id (and maybe cluster_title)
TEXT_COL = "text_for_model" 

K = int(dfB["cluster_id"].nunique())
print("K:", K)

S_PER_CLUSTER = 6000   # 5 clusters -> ~30k points, good for interactive 3D
SEED = 42

rng = np.random.default_rng(SEED)
sample_idx = []

for cid in sorted(dfB["cluster_id"].unique()):
    idx = dfB.index[dfB["cluster_id"] == cid].to_numpy()
    take = min(S_PER_CLUSTER, len(idx))
    pick = rng.choice(idx, size=take, replace=False)
    sample_idx.append(pick)

sample_idx = np.concatenate(sample_idx)
viz_df = dfB.loc[sample_idx, ["product_id", TEXT_COL, "cluster_id"]].copy()

# If you created business names / auto titles earlier:
if "cluster_title" in dfB.columns:
    viz_df["cluster_title"] = dfB.loc[sample_idx, "cluster_title"].values
elif "cluster_title_auto" in dfB.columns:
    viz_df["cluster_title"] = dfB.loc[sample_idx, "cluster_title_auto"].values
else:
    viz_df["cluster_title"] = viz_df["cluster_id"].astype(str)

# Short hover text
viz_df["hover_text"] = viz_df[TEXT_COL].astype(str).str.slice(0, 120)

viz_df.head()


K: 5


,product_id,text_for_model,cluster_id,cluster_title,hover_text
24130,110581531,2006 model citroen picasso için tavan araç üst...,0,0,2006 model citroen picasso için tavan araç üst...
213282,74362312,samsung galaxy j7 prime kulaklık oyuncu kulakl...,0,0,samsung galaxy j7 prime kulaklık oyuncu kulakl...
298588,53622835,usb bellek 16 gb roller kalem set kutulu,0,0,usb bellek 16 gb roller kalem set kutulu
252693,40564461,balık sırtı anten gri 100 bakır kablolu lastik...,0,0,balık sırtı anten gri 100 bakır kablolu lastik...
76394,96797727,direct drive otomatik punteriz makinesi bt 290...,0,0,direct drive otomatik punteriz makinesi bt 290...


In [11]:

{0: 'seti / kedi / yağı / bakım / doğal / saç / kremi / fırçası / temizleme / sprey',
 1: 'ahşap / takımı / metal / seti / dekoratif / mutfak / gold / masa / gümüş / banyo',
 2: 'makinesi / seti / kalem / led / pro / oyuncu / usb / oto / kırmızı / model',
 3: 'bebek / çocuk / kişilik / seti / yatak / örtüsü / takımı / odası / çift / tek kişilik',
 4: 'kadın / erkek / beden / büyük beden / büyük / deri / detaylı / takım / tesettür / çantası'}

{0: 'seti / kedi / yağı / bakım / doğal / saç / kremi / fırçası / temizleme / sprey',
 1: 'ahşap / takımı / metal / seti / dekoratif / mutfak / gold / masa / gümüş / banyo',
 2: 'makinesi / seti / kalem / led / pro / oyuncu / usb / oto / kırmızı / model',
 3: 'bebek / çocuk / kişilik / seti / yatak / örtüsü / takımı / odası / çift / tek kişilik',
 4: 'kadın / erkek / beden / büyük beden / büyük / deri / detaylı / takım / tesettür / çantası'}

In [12]:
# Edit these after you inspect keywords/examples
business_names = {
    0: "Elektronik & Otomotiv Aksesuarları",
    1: "Kozmetik & Kişisel Bakım / Ev Bakım",
    2: "Ev & Yaşam",
    3: "Anne & Bebek & Oyuncak",
    4: "Moda"
}

dfB["cluster_title"] = dfB["cluster_id"].map(business_names).fillna(dfB["cluster_id"].map(cluster_titles))
dfB[["cluster_id","cluster_title"]].drop_duplicates().sort_values("cluster_id")


,cluster_id,cluster_title
0,0,Elektronik & Otomotiv Aksesuarları
300,1,Kozmetik & Kişisel Bakım / Ev Bakım
569,2,Ev & Yaşam
1552,3,Anne & Bebek & Oyuncak
600,4,Moda


In [13]:
import os

# Set these if not in your namespace
EMB_PATH = "embeddings_float16.memmap"  # same as before


N = len(dfB)
emb = np.memmap(EMB_PATH, dtype="float16", mode="r", shape=(N, dim))

# Extract sample embeddings (float32 for UMAP)
X_viz = emb[sample_idx].astype(np.float32)
print("X_viz:", X_viz.shape, "dtype:", X_viz.dtype)


X_viz: (30000, 768) dtype: float32


In [14]:
import umap

umap_3d = umap.UMAP(
    n_components=3,
    n_neighbors=30,
    min_dist=0.15,
    metric="cosine",
    random_state=SEED
)

X_3d = umap_3d.fit_transform(X_viz)
viz_df["x"] = X_3d[:, 0]
viz_df["y"] = X_3d[:, 1]
viz_df["z"] = X_3d[:, 2]

viz_df.head()


c:\Users\ilker\anaconda3\envs\torchgpu\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


,product_id,text_for_model,cluster_id,cluster_title,hover_text,x,y,z
24130,110581531,2006 model citroen picasso için tavan araç üst...,0,0,2006 model citroen picasso için tavan araç üst...,5.508156,8.869628,3.121411
213282,74362312,samsung galaxy j7 prime kulaklık oyuncu kulakl...,0,0,samsung galaxy j7 prime kulaklık oyuncu kulakl...,1.865135,7.256720,8.011600
298588,53622835,usb bellek 16 gb roller kalem set kutulu,0,0,usb bellek 16 gb roller kalem set kutulu,1.998621,9.581382,6.263143
252693,40564461,balık sırtı anten gri 100 bakır kablolu lastik...,0,0,balık sırtı anten gri 100 bakır kablolu lastik...,3.638835,9.642711,5.971848
76394,96797727,direct drive otomatik punteriz makinesi bt 290...,0,0,direct drive otomatik punteriz makinesi bt 290...,5.695301,10.057677,6.067677


In [15]:
dfB

,product_id,text_for_model,cluster_id,cluster_title
0,2925783,ac18 4port gigabit 1900mbps usb3 ac router,0,Elektronik & Otomotiv Aksesuarları
1,69822182,pfs3110 8et 96 8 port poe switch yönetilmeyen,0,Elektronik & Otomotiv Aksesuarları
2,2573924,des 1016c 16 port 10 100mbps yönetilemez metal...,0,Elektronik & Otomotiv Aksesuarları
3,68891704,digitus 19 inch 25 port cat 3 isdn patch panel...,0,Elektronik & Otomotiv Aksesuarları
4,80660199,vmg1312 t20b vdsl2 adsl2 4 port kablosuz usb d...,0,Elektronik & Otomotiv Aksesuarları
...,...,...,...,...
341995,175616692,piramit hava bahçe şöminesi ateş çukuru özel t...,1,Kozmetik & Kişisel Bakım / Ev Bakım
341996,165257446,eskitme kum saati şömine maşa takımı,1,Kozmetik & Kişisel Bakım / Ev Bakım
341997,175585785,romantik hava bahçe şöminesi ateş çukuru özel ...,1,Kozmetik & Kişisel Bakım / Ev Bakım
341998,65640160,şömine soba topuz başlı 4 lü eskitme maşa takımı,1,Kozmetik & Kişisel Bakım / Ev Bakım


In [16]:
import pandas as pd
import numpy as np
import json
# pick leaf label column
LEAF_COL = "category_fixed"


dfC = dfB[["product_id", "cluster_id", "cluster_title"]].merge(
    train_df[["product_id", LEAF_COL]],
    on="product_id",
    how="left"
)
category_mapping = json.load(open('category_map.json', 'r', encoding='utf-8'))

# main category derived from the fixed leaf labels
dfC["main_category_fixed"] = dfC[LEAF_COL].astype(str).map(category_mapping).fillna("Diğer")

print("Rows in dfC:", len(dfC))
print("Missing leaf labels after merge:", dfC[LEAF_COL].isna().sum())
dfC.head()


Rows in dfC: 588486
Missing leaf labels after merge: 0


,product_id,cluster_id,cluster_title,category_fixed,main_category_fixed
0,2925783,0,Elektronik & Otomotiv Aksesuarları,ADSL Modemler,Elektronik
1,2925783,0,Elektronik & Otomotiv Aksesuarları,ADSL Modemler,Elektronik
2,69822182,0,Elektronik & Otomotiv Aksesuarları,ADSL Modemler,Elektronik
3,69822182,0,Elektronik & Otomotiv Aksesuarları,ADSL Modemler,Elektronik
4,69822182,0,Elektronik & Otomotiv Aksesuarları,ADSL Modemler,Elektronik


In [17]:
cluster_sizes = dfC.groupby("cluster_title").size().rename("cluster_n")

main_counts = (dfC.groupby(["cluster_title", "main_category_fixed"])
                 .size()
                 .rename("cnt")
                 .reset_index())

main_counts = main_counts.merge(cluster_sizes.reset_index(), on="cluster_title", how="left")
main_counts["pct"] = main_counts["cnt"] / main_counts["cluster_n"]

# pivot to wide (%)
main_pivot_pct = (main_counts.pivot_table(
    index="cluster_title",
    columns="main_category_fixed",
    values="pct",
    fill_value=0.0
).sort_index())

display(main_pivot_pct)


main_category_fixed,Aksesuar,Anne & Bebek,Ayakkabı,Elektronik,Ev & Yaşam,Giyim,Kozmetik,Otomotiv,Oyuncak,Spor
cluster_title,,,,,,,,,,
Anne & Bebek & Oyuncak,0.019121,0.082381,0.001013,0.004040,0.757764,0.005146,0.077104,0.008054,0.021609,0.023767
Elektronik & Otomotiv Aksesuarları,0.040186,0.010832,0.026853,0.253679,0.416735,0.019897,0.008621,0.140377,0.039016,0.043805
Ev & Yaşam,0.060050,0.043123,0.030417,0.017799,0.380921,0.022166,0.327245,0.021823,0.050313,0.046143
Kozmetik & Kişisel Bakım / Ev Bakım,0.007343,0.093612,0.000665,0.000638,0.842212,0.006112,0.001609,0.003649,0.035764,0.008395
Moda,0.119230,0.019139,0.045174,0.000910,0.022835,0.770274,0.005187,0.000272,0.002470,0.014509


In [18]:
top_main = (main_counts.sort_values(["cluster_id", "pct"], ascending=[True, False])
                      .groupby("cluster_id")
                      .head(10)
                      .copy())

top_main["pct"] = (top_main["pct"] * 100).round(2)
display(top_main[["cluster_id", "main_category_fixed", "cnt", "pct", "cluster_n"]])


KeyError: 'cluster_id'

In [ ]:
leaf_counts = (dfC.groupby(["cluster_id", LEAF_COL])
                 .size()
                 .rename("cnt")
                 .reset_index())

leaf_counts = leaf_counts.merge(cluster_sizes.reset_index(), on="cluster_id", how="left")
leaf_counts["pct"] = leaf_counts["cnt"] / leaf_counts["cluster_n"]

TOP_N = 20
top_leaf = (leaf_counts.sort_values(["cluster_id", "pct"], ascending=[True, False])
                      .groupby("cluster_id")
                      .head(TOP_N)
                      .copy())

top_leaf["pct"] = (top_leaf["pct"] * 100).round(3)
display(top_leaf[["cluster_id", LEAF_COL, "cnt", "pct", "cluster_n"]])


,cluster_id,category_fixed,cnt,pct,cluster_n
443,0,Masaj Yastığı,8492,7.134,119035
186,0,Dolap İçi Düzenleyici,1296,1.089,119035
221,0,Endüstriyel Temizleme,1027,0.863,119035
41,0,Araç İçi Kameralar,738,0.620,119035
189,0,Doğrayıcı&Rondo,686,0.576,119035
...,...,...,...,...,...
3580,4,İş Kıyafetleri,1305,0.758,172185
3448,4,Spor Terlik,1244,0.722,172185
3498,4,Tesettür Hırka,1222,0.710,172185
2995,4,Babydoll,1156,0.671,172185


In [ ]:
top_leaf[top_leaf["cluster_id"] == 0]

,cluster_id,category_fixed,cnt,cluster_n,pct
443,0,Masaj Yastığı,8492,119035,7.134
186,0,Dolap İçi Düzenleyici,1296,119035,1.089
221,0,Endüstriyel Temizleme,1027,119035,0.863
41,0,Araç İçi Kameralar,738,119035,0.620
189,0,Doğrayıcı&Rondo,686,119035,0.576
694,0,Tablet,638,119035,0.536
156,0,Cilt Bakım Aletleri,596,119035,0.501
569,0,Pikap & Gramofon,593,119035,0.498
15,0,Akıllı Bileklik,587,119035,0.493
702,0,Tansiyon Aleti,585,119035,0.491


In [ ]:
# main-category share per cluster
tmp = main_counts.copy()
tmp["p"] = tmp["pct"]

# entropy (lower = purer)
entropy = (tmp.groupby("cluster_id")["p"]
             .apply(lambda p: float(-(p * np.log(p + 1e-12)).sum()))
             .rename("main_entropy"))

top_share = (tmp.groupby("cluster_id")["p"].max().rename("main_top_share"))

summary = pd.concat([cluster_sizes, top_share, entropy], axis=1).reset_index()
summary["main_top_share"] = (summary["main_top_share"] * 100).round(2)
summary["main_entropy"] = summary["main_entropy"].round(3)

display(summary.sort_values("cluster_id"))


,cluster_id,cluster_n,main_top_share,main_entropy
0,0,119035,40.79,1.640
1,1,127770,55.69,1.161
2,2,108151,94.46,0.319
3,3,61073,37.65,1.623
4,4,172185,73.19,0.964


In [ ]:
import plotly.express as px

fig = px.scatter_3d(
    viz_df,
    x="x", y="y", z="z",
    color="cluster_title",
    hover_data={"product_id": True, "hover_text": True, "cluster_id": True, "x": False, "y": False, "z": False},
    opacity=0.65,
    title=f"UMAP 3D projection of embeddings (sample={len(viz_df):,})"
)

fig.update_traces(marker=dict(size=3))
fig.update_layout(
    legend_title_text="Cluster",
    scene=dict(
        xaxis_title="UMAP-1",
        yaxis_title="UMAP-2",
        zaxis_title="UMAP-3"
    ),
    height=750
)

fig.show()


# Evaluation

In [ ]:
import pandas as pd
import numpy as np

size_tbl = (dfB["cluster_id"].value_counts()
            .rename_axis("cluster_id")
            .reset_index(name="count"))
size_tbl["pct"] = (size_tbl["count"] / size_tbl["count"].sum() * 100).round(2)

# balance indicators
p = size_tbl["count"].values / size_tbl["count"].sum()
size_tbl["balance_entropy"] = -np.sum(p * np.log(p + 1e-12))  # same value repeated; keep separate if you want
size_tbl


,cluster_id,count,pct,balance_entropy
0,1,83160,24.32,1.579319
1,0,81823,23.92,1.579319
2,4,73173,21.40,1.579319
3,2,64499,18.86,1.579319
4,3,39345,11.50,1.579319


In [ ]:
import numpy as np
import pandas as pd

# centroids: (K, dim) float32, embeddings normalized
C = centroids.astype(np.float32)
C = C / (np.linalg.norm(C, axis=1, keepdims=True) + 1e-12)

sim = C @ C.T  # cosine sim
sep = pd.DataFrame(sim, index=[f"C{i}" for i in range(len(C))],
                        columns=[f"C{i}" for i in range(len(C))])
sep


,C0,C1,C2,C3,C4
C0,1.000000,0.685339,0.696584,0.532553,0.685704
C1,0.685339,1.000000,0.709527,0.594832,0.677258
C2,0.696584,0.709527,1.000000,0.674297,0.693329
C3,0.532553,0.594832,0.674297,1.000000,0.593344
C4,0.685704,0.677258,0.693329,0.593344,1.000000


In [ ]:
import numpy as np
from tqdm.auto import tqdm
import faiss

# assumes you have: EMB_PATH, N, dim, centroids, dfB
emb = np.memmap(EMB_PATH, dtype="float16", mode="r", shape=(len(dfB), dim))

index = faiss.IndexFlatIP(dim)
index.add(centroids.astype(np.float32))

B = 200_000
margin = np.empty(len(dfB), dtype=np.float32)
top1 = np.empty(len(dfB), dtype=np.int16)

for start in tqdm(range(0, len(dfB), B), desc="Margin(top1-top2)"):
    end = min(start+B, len(dfB))
    Xb = emb[start:end].astype(np.float32)
    sims, ids = index.search(Xb, 2)              # top-2
    top1[start:end] = ids[:, 0].astype(np.int16)
    margin[start:end] = (sims[:, 0] - sims[:, 1]).astype(np.float32)

dfB["cluster_margin"] = margin
dfB["cluster_id_check"] = top1  # should match cluster_id if you assigned earlier with same centroids

# ambiguity summary
q = dfB["cluster_margin"].quantile([0.05,0.10,0.25,0.50,0.75,0.90,0.95]).to_frame("margin_quantile")
q


Margin(top1-top2): 100%|██████████| 2/2 [00:01<00:00,  1.99it/s]


,margin_quantile
0.05,0.004503
0.10,0.009327
0.25,0.024870
0.50,0.052594
0.75,0.081937
0.90,0.103579
0.95,0.115534


In [ ]:
thr = float(dfB["cluster_margin"].quantile(0.10))
dfB["is_ambiguous"] = dfB["cluster_margin"] <= thr
amb_rate = dfB["is_ambiguous"].mean()
print("Ambiguous rate (bottom 10% margin):", round(amb_rate*100, 2), "%  | threshold:", thr)


Ambiguous rate (bottom 10% margin): 10.0 %  | threshold: 0.009326985478401184


In [ ]:
from sklearn.metrics import silhouette_score

# Use the same sample you used for UMAP, or create a new one
# X_viz: (n_sample, dim) float32 normalized embeddings
# viz_df contains cluster_id for those points

sil = silhouette_score(X_viz, viz_df["cluster_id"].values, metric="cosine")
print("Silhouette (cosine, sample):", sil)


Silhouette (cosine, sample): 0.06374754011631012


In [ ]:
LEAF_COL = "category_fixed" if "category_fixed" in train_df.columns else "category"

dfC = dfB[["product_id", "cluster_id"]].merge(
    train_df[["product_id", LEAF_COL]],
    on="product_id",
    how="left"
)
dfC["main_category_fixed"] = dfC[LEAF_COL].astype(str).map(category_mapping).fillna("Diğer")

# distribution
cnt = (dfC.groupby(["cluster_id", "main_category_fixed"]).size().rename("cnt").reset_index())
tot = dfC.groupby("cluster_id").size().rename("n").reset_index()
cnt = cnt.merge(tot, on="cluster_id")
cnt["p"] = cnt["cnt"] / cnt["n"]

# summary metrics
import numpy as np
summary = (cnt.groupby("cluster_id")
           .apply(lambda g: pd.Series({
               "n": int(g["n"].iloc[0]),
               "top_main": g.sort_values("p", ascending=False)["main_category_fixed"].iloc[0],
               "top_main_share": float(g["p"].max()),
               "main_entropy": float(-(g["p"] * np.log(g["p"] + 1e-12)).sum())
           }))
           .reset_index())

summary["top_main_share"] = (summary["top_main_share"]*100).round(2)
summary["main_entropy"] = summary["main_entropy"].round(3)
summary.sort_values("cluster_id")


,cluster_id,n,top_main,top_main_share,main_entropy
0,0,119035,Ev & Yaşam,40.79,1.640
1,1,127770,Ev & Yaşam,55.69,1.161
2,2,108151,Ev & Yaşam,94.46,0.319
3,3,61073,Ev & Yaşam,37.65,1.623
4,4,172185,Giyim,73.19,0.964


In [ ]:
leaf_cnt = (dfC.groupby(["cluster_id", LEAF_COL]).size().rename("cnt").reset_index())
leaf_tot = dfC.groupby("cluster_id").size().rename("n").reset_index()
leaf_cnt = leaf_cnt.merge(leaf_tot, on="cluster_id")
leaf_cnt["p"] = leaf_cnt["cnt"] / leaf_cnt["n"]

# Herfindahl index and effective number of categories
leaf_summary = (leaf_cnt.groupby("cluster_id")
                .apply(lambda g: pd.Series({
                    "n": int(g["n"].iloc[0]),
                    "herfindahl": float((g["p"]**2).sum()),
                    "effective_leaf_categories": float(1.0 / ((g["p"]**2).sum() + 1e-12)),
                    "top_leaf_share": float(g["p"].max())
                }))
                .reset_index())

leaf_summary["top_leaf_share"] = (leaf_summary["top_leaf_share"]*100).round(2)
leaf_summary["herfindahl"] = leaf_summary["herfindahl"].round(4)
leaf_summary["effective_leaf_categories"] = leaf_summary["effective_leaf_categories"].round(1)
leaf_summary.sort_values("cluster_id")


,cluster_id,n,herfindahl,effective_leaf_categories,top_leaf_share
0,0,119035.0,0.0077,129.1,7.13
1,1,127770.0,0.0038,260.0,1.85
2,2,108151.0,0.0210,47.5,12.05
3,3,61073.0,0.0120,83.2,8.34
4,4,172185.0,0.0307,32.6,13.10


In [ ]:
pivot = pd.crosstab(dfC["cluster_id"], dfC["main_category_fixed"], normalize="index") * 100
pivot = pivot.round(2)
pivot


main_category_fixed,Aksesuar,Anne & Bebek,Ayakkabı,Elektronik,Ev & Yaşam,Giyim,Kozmetik,Otomotiv,Oyuncak,Spor
cluster_id,,,,,,,,,,
0,3.77,0.25,1.84,25.86,40.79,0.97,3.50,14.20,3.46,5.37
1,1.39,4.52,0.83,0.37,55.69,0.30,32.14,1.46,0.79,2.51
2,0.73,0.54,0.07,0.17,94.46,0.07,0.69,0.53,1.52,1.21
3,3.69,28.80,2.16,0.72,37.65,6.15,0.60,1.04,15.78,3.41
4,13.97,1.03,5.56,0.24,3.19,73.19,0.94,0.07,0.24,1.57
